<div style="text-align: justify; max-width: 90ch">

# Lecture 01d: LangChain chains

**Status: Mandatory reading.**

In `lec_01c` you wrote `build_prompt`, `ask`, and `tag_question` by hand. Every LLM application has those same three parts, so libraries exist that name them and give them one common interface.
[**LangChain**](https://docs.langchain.com/oss/python/langchain/overview) is the most widely used one; section 6 records why this course chose it.

**What this notebook covers**

- Why a framework: what plain client code runs into as a system grows.
- LangChain's parts: chat model, prompt template, output parser, and the `|` that chains them.
- `invoke` for one input, `batch` for many, structured output inside a chain.
- Reading the prompt a template renders, and swapping the local model for a hosted one in one line.
- What LangChain adds, what it costs, and when the plain client is enough.
- Why this course uses LangChain: a decision record, like the model's in `lec_01b`.

</div>

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # this notebook runs from reading_material/
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
os.environ.setdefault(
    "HF_HUB_DISABLE_PROGRESS_BARS", "1"
)  # no download bars in saved outputs
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-01-basics"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

## 1. Why a framework

Your `ask` function from `lec_01c` works, but it is written for one client, `ollama`. Three problems appear as a system grows:

- **Every provider has its own client.** Ollama, OpenAI and Anthropic name the call, the messages and the reply differently,
  so changing model means rewriting every call.
- **The same glue code comes back** in every project: filling prompts, parsing replies, running many questions, streaming, retrying after a timeout.
- **Steps must pass data to each other.** Lecture 2 puts retrieval before the model; lecture 3 adds decisions and loops.

A **framework** is a library that solves these once: it defines the parts and how they connect, and you supply the prompts and the model.

</div>

<div style="text-align: justify; max-width: 90ch">

LangChain names the parts you already wrote:

| LangChain name | What you wrote in `lec_01c` | Job |
| --- | --- | --- |
| **chat model** (`ChatOllama`) | `client.chat(...)` | sends messages to a model, returns a message |
| **prompt template** (`ChatPromptTemplate`) | `build_prompt(question, style)` | fills named slots into messages |
| **output parser** (`StrOutputParser`) | `.message.content.strip()` | turns the model's message into what your code wants |
| **chain** (built with `\|`) | calling them in order | runs the parts in order as one object |

Every part, and every chain built from parts, is a **Runnable**, LangChain's [common interface](https://reference.langchain.com/python/langchain-core/runnables),
with the same methods on every object:

- `invoke` for one input
- `batch` for a list
- `stream` for the answer piece by piece as it is generated (`lec_01c`)

Section 5 weighs what this buys against what it costs.

</div>

<div style="text-align: justify; max-width: 90ch">

A **chat model** wraps one provider's client. `ChatOllama` comes from `langchain-ollama`, one of LangChain's **partner packages**:
each provider's integration is a separate package, so a project installs only the ones it uses ([chat model integrations](https://docs.langchain.com/oss/python/integrations/chat)).
The arguments are settings you know from `lec_01a` and `lec_01c`:

- `model` and `base_url`: which model, on which Ollama server.
- `reasoning=False`: thinking off, like `think=False`.
- `temperature=0`: the most likely token every time, so answers repeat.
- `num_predict=200`: at most 200 tokens of answer; Ollama's name for what other providers call `max_tokens`.

</div>

In [3]:
llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,  # the same switch as think=False in the ollama client
    temperature=0,
    num_predict=200,
)

reply = llm.invoke("In one sentence, what is a DataFrame?")
print(reply.content)

A DataFrame is a two-dimensional data structure used in Python's pandas library to store and manipulate tabular data, such as rows and columns, with support for various data types and operations.


In [4]:
reply.usage_metadata  # tokens in and out, the same numbers lec_01a computed by hand

{'input_tokens': 25, 'output_tokens': 38, 'total_tokens': 63}

<div style="text-align: justify; max-width: 90ch">

`invoke` returns an `AIMessage`: the `assistant` message from `lec_01c`, as an object. LangChain has one message type per role, and templates (section 2) name the roles with short strings:

| Role in `lec_01c` | LangChain type | Name in a template |
| --- | --- | --- |
| `system` | `SystemMessage` | `"system"` |
| `user` | `HumanMessage` | `"human"` |
| `assistant` | `AIMessage` | `"ai"` |

An `AIMessage` carries:

- `content`: the text.
- `usage_metadata`: the token counts above.
- `response_metadata`: what the provider added, such as the model name and `done_reason` (`lec_01a`).

</div>

<div style="text-align: justify; max-width: 90ch">

## 2. Prompt templates

A **prompt template** is a prompt with **named slots** in curly braces, filled in at run time; it replaces your `build_prompt` function.
Two ways to build one:

- `ChatPromptTemplate.from_messages` takes a list of `(role, text)` pairs, one per message.
- `ChatPromptTemplate.from_template` takes one string and makes a single human message; the brace examples below use it.

`invoke` with a dict fills the slots and returns a **prompt value**, the rendered messages, without calling any model, so you can look at exactly what will be sent.
The dict needs one key per slot; `prompt.input_variables` lists them.

</div>

In [5]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a teaching assistant for a Python course. Answer in at most {sentences} sentences.",
        ),
        ("human", "{question}"),
    ]
)

rendered = prompt.invoke(
    {"sentences": 2, "question": "What does the axis argument mean in pandas drop()?"}
)
for message in rendered.messages:
    print(f"[{message.type}] {message.content}")

[system] You are a teaching assistant for a Python course. Answer in at most 2 sentences.
[human] What does the axis argument mean in pandas drop()?


<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

Read the two rendered messages. That is the prompt the model will *actually* receive: the system message with `{sentences}` filled in, then the question.
Nobody typed that text; the template produced it. When an answer looks wrong, read the rendered prompt before blaming the model.
In lecture 2, where a system has several steps, MLflow tracing records the input and output of every step the same way.

</div>

<div style="text-align: justify; max-width: 90ch">

Curly braces are the slot syntax, which bites the first time a template contains literal braces, for example a JSON example for the model.

</div>

In [6]:
# WRONG: the JSON example's braces are read as slots named '"category"'
broken = ChatPromptTemplate.from_template(
    'Reply as JSON like {"category": "pandas"}. Question: {question}'
)
try:
    broken.invoke({"question": "How do I merge tables?"})
except KeyError as error:
    print("KeyError:", error)

KeyError: 'Input to ChatPromptTemplate is missing variables {\'"category"\'}.  Expected: [\'"category"\', \'question\'] Received: [\'question\']\nNote: if you intended {"category"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"category"}}\'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT '


In [7]:
# RIGHT: double the braces you want to keep
fixed = ChatPromptTemplate.from_template(
    'Reply as JSON like {{"category": "pandas"}}. Question: {question}'
)
print(fixed.invoke({"question": "How do I merge tables?"}).messages[0].content)

Reply as JSON like {"category": "pandas"}. Question: How do I merge tables?


<div style="text-align: justify; max-width: 90ch">

## 3. Chains

`|` joins parts into a **chain**: the output of each becomes the input of the next. This notation is the **LangChain Expression Language** (LCEL),
and the chain it builds is itself a Runnable, with the same `invoke`, `batch` and `stream` as its parts. `prompt | llm | StrOutputParser()` reads as "fill the template, send it, keep the text":

| Step | Receives | Returns |
| --- | --- | --- |
| `prompt` | a dict of slot values | a prompt value (messages) |
| `llm` | messages | an `AIMessage` |
| `StrOutputParser()` | an `AIMessage` | its `content`, a `str` |

When a chain fails, check that each step accepts what the one before it returns.

</div>

In [8]:
chain = prompt | llm | StrOutputParser()

print(
    chain.invoke(
        {
            "sentences": 2,
            "question": "Why do we scale features after the train/test split?",
        }
    )
)

We scale features after the train/test split to ensure that features of different scales do not disproportionately influence the model. This helps improve the performance and stability of many machine learning algorithms.


<div style="text-align: justify; max-width: 90ch">

`batch` runs the chain over a list of inputs and returns the answers in the same order. **Concurrency** is how many requests run at the same time.
Ollama on a laptop serves one request at a time, so we cap it at 1 with `max_concurrency`; a hosted model would take 10 in parallel with the same line.

</div>

In [9]:
questions = [
    "What is the difference between loc and iloc?",
    "When is a silhouette score useful?",
    "What does random_state do?",
]
inputs = [{"sentences": 1, "question": q} for q in questions]

answers = chain.batch(inputs, config={"max_concurrency": 1})
pd.DataFrame({"question": questions, "answer": answers})

,question,answer
0,What is the difference between loc and iloc?,"`loc` is used for label-based indexing (e.g., ..."
1,When is a silhouette score useful?,A silhouette score is useful for evaluating th...
2,What does random_state do?,The `random_state` parameter controls the rand...


<div style="text-align: justify; max-width: 90ch">

Structured output from `lec_01c` is a method on the chat model: `with_structured_output` takes the same JSON schema and returns a new chat model whose answers arrive as a dict.
`method="json_schema"`, the default, uses the provider's own schema mode: for Ollama, the `format=` argument from `lec_01c`,
so the answer has the schema's shape, unless `num_predict` cuts it off (`lec_01c`). The new model slots into a chain like any other part.

</div>

In [10]:
TAG_SCHEMA = {
    "title": "ForumTag",
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": ["pandas", "plots", "sklearn", "python-basics", "other"],
        },
        "needs_code": {"type": "boolean"},
    },
    "required": ["category", "needs_code"],
}

tag_prompt = ChatPromptTemplate.from_template(
    "Tag this Python-course forum question.\n\nQuestion: {question}"
)
tag_chain = tag_prompt | llm.with_structured_output(TAG_SCHEMA, method="json_schema")

tag_chain.invoke({"question": "How do I change the colours in a seaborn heatmap?"})

{'category': 'python-basics', 'needs_code': True}

<div style="text-align: justify; max-width: 90ch">

The chain returns a dict, but the tag is wrong: a seaborn heatmap is `plots`, not `python-basics`. This `tag_prompt` leaves out the category definitions,
and `lec_01c` showed that without them a small model tags most questions as `python-basics`. A chain changes how the parts connect, not what the prompt says.

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. Swapping the model is one line

Everything above used `ChatOllama`. Many providers accept the request format of OpenAI's API; they are called **OpenAI-compatible**,
and Ollama is one of them ([docs](https://docs.ollama.com/api/openai-compatibility)). One chat model, `ChatOpenAI` from the `langchain-openai` partner package, talks to all of them.
A hosted model, for a job where privacy allows it and latency matters, is the same chain with a different first line:

```python
from langchain_openai import ChatOpenAI          # any OpenAI-compatible provider
llm = ChatOpenAI(model="...", base_url="...", api_key=os.environ["PROVIDER_KEY"])
chain = prompt | llm | StrOutputParser()        # unchanged
```

The key belongs in `.env`, like the Hugging Face token. The optional `lec_01f` tries it against a hosted endpoint.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. What LangChain adds, and what it costs

Everything in this notebook could be written with the `ollama` client in a few more lines. The framework pays off as the system grows:

- **One interface for every part.** Prompts, models, parsers and chains are all Runnables, so `invoke`, `batch` and `stream` work the same on each;
  you learn them once.
- **Provider independence.** The chain does not know which model it calls; section 4 changed provider in one line.
- **Ready-made integrations** for hundreds of tools ([providers](https://docs.langchain.com/oss/python/integrations/providers/overview)):
  document loaders, text splitters, embedding models, vector stores. Lecture 2 builds retrieval from three of them.

</div>

<div style="text-align: justify; max-width: 90ch">

It also brings:

- **Helpers for failures.** `with_retry` sends a failed call again; `with_fallbacks` switches to a second model when the first one fails.
- **Observability**, seeing what each step received and returned. Every Runnable reports its inputs and outputs; lecture 2 turns these reports into MLflow traces with one line.
- **Multi-step systems.** [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview), from the same team, builds systems that decide and loop,
  from the same parts; lecture 3 uses it.

</div>

<div style="text-align: justify; max-width: 90ch">

The costs are real too:

- **Another layer between you and the prompt.** The template writes the text the model reads; section 2's habit, reading the rendered prompt, is the defence.
- **A fast-moving API.** LangChain 1.0 moved older classes such as `LLMChain` to a separate `langchain-classic` package ([migration guide](https://docs.langchain.com/oss/python/migrate/langchain-v1)),
  and many online tutorials still use them. This course pins versions in `uv.lock` and uses only the `|` style.
- **Longer error messages.** A failure inside a chain prints a stack trace through library code; look for the lines that point into your notebook.
- **More dependencies** to install and keep up to date.

</div>

<div style="text-align: justify; max-width: 90ch">

- **When the plain client is enough:** one model, one prompt, no plan to switch. A nightly script that tags forum questions is clearer with the `ollama` client from `lec_01c`.
- **When a framework pays:** several steps, a model you may change, or retrieval.

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. Which framework: a decision record

`lec_01b` chose the model from requirements, not from popularity or size. A framework deserves the same treatment: lectures 2 and 3 are written on top of it, so switching later means rewriting them.
The requirements come from who this course is for:

| Requirement | Why it matters here |
| --- | --- |
| **Your own step is a plain function** | the Python course did not teach classes or type hints; lecture 2's `format_context` is a function |
| **The prompt stays in your hands** | section 2's habit: read the exact text the model receives |
| **Local integrations** | Ollama, Chroma and pgvector, without writing the glue yourself |
| **Retrieval and agents in one family** | lecture 2 retrieves, lecture 3 decides and loops; one way of working for both |

</div>

<div style="text-align: justify; max-width: 90ch">

Five candidates against those requirements, starting with no framework at all: the `ollama` client of `lec_01c`, with `chromadb` for retrieval.

| Requirement | plain `ollama` | LangChain | LlamaIndex | Haystack | DSPy |
| --- | --- | --- | --- | --- | --- |
| step is a function | yes | yes | no: [Workflow classes](https://developers.llamaindex.ai/python/framework/module_guides/workflow/) | no: [component classes](https://docs.haystack.deepset.ai/docs/custom-components) | no: [Module classes](https://dspy.ai/learn/programming/modules/) |
| prompt in your hands | yes | yes | partly: [default prompts](https://developers.llamaindex.ai/python/framework/module_guides/models/prompts/) | yes | no: [built from a signature](https://dspy.ai/learn/programming/signatures/) |
| local integrations | write them | yes | [yes](https://llamahub.ai/) | [yes](https://haystack.deepset.ai/integrations) | models only |
| retrieval and agents | write them | yes, with [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) | yes | yes | yes |

Two more requirements decide nothing, because every framework meets them: an open license (MIT; Apache-2.0 for Haystack) and one-line tracing in MLflow ([integrations](https://mlflow.org/docs/latest/genai/tracing/integrations/)).
Plain `ollama` is traced by hand, with `@mlflow.trace` in `lec_02b`.

</div>

<div style="text-align: justify; max-width: 90ch">

The record:

- **Decision:** LangChain 1.x, the `|` style only; LangGraph for lecture 3's agents.
- **Requirements:** the four rows above.
- **Candidates checked:**
  - plain `ollama` with `chromadb` (everything visible, but every integration and the agent loop are yours to write)
  - LlamaIndex (default prompts; a custom step is a Workflow class)
  - Haystack (the closest rival: explicit pipelines and visible prompts, but a custom step is a class with decorators and types)
  - DSPy (writes the prompt for you, which is its point; optional in lecture 3)
- **Evidence:**
  - sections 2-5 of this notebook
  - `lec_02b` as the no-framework baseline
  - secondary, the largest community: about 147,000 GitHub stars for [LangChain](https://github.com/langchain-ai/langchain),
    52,000 for [LlamaIndex](https://github.com/run-llama/llama_index), 27,000 for [Haystack](https://github.com/deepset-ai/haystack) (October 2026),
    so more tutorials and more of the code you will meet at work
- **Revisit when:** LangChain 2.0 breaks the API (1.x promises no breaking changes, [release policy](https://docs.langchain.com/oss/python/release-policy)),
  or the course drops agents and plain `ollama` with `chromadb` is enough.

</div>

<div style="text-align: justify; max-width: 90ch">

### ⏸ Pause point

Think of a system you might build for the final assignment. Would you write it with LangChain or with the plain client? Name the item from section 5's lists, or the row of section 6's table, that decides it.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- A framework solves once what every LLM project needs: one interface over many providers, the glue between steps, and hooks to see inside.
- LangChain names the parts you already wrote: **chat model**, **prompt template**, **output parser**; `|` (LCEL) chains them into a **Runnable** with `invoke`, `batch`, `stream`.
- Templates render without calling the model; literal braces must be doubled.
- `with_structured_output` puts a JSON schema inside a chain.
- Read the rendered prompt before blaming the model: it is exactly what the model receives.
- The price is a layer between you and the prompt and a fast-moving API; for one model and one prompt, the plain client is enough.
- A framework is chosen like a model, with a decision record: LangChain keeps steps as plain functions and the prompt visible.

**Next: lecture 2**, where the model stops answering from memory and starts answering from *your* documents: embeddings, retrieval, and a hand-built RAG, with MLflow tracing to see inside each step.

</div>